# Generative Question Answering with T5/BART and SQAC Spanish (using Hugging Face)

Author: Vladimir Araujo

Based on: https://www.spark64.com/post/machine-comprehension



## Instrucciones Generales

El siguiente práctico se realiza individualmente. El formato de entregar es el **archivo .ipynb con todas las celdas ejecutadas**. Todas las preguntas deben ser respondida en celdas de texto. No se aceptará el _output_ de una celda de código como respuesta.

**Nombre:** COMPLETAR

El siguiente práctico cuanta con 2 secciones donde cada una contendrá 1 o más actividades a realizar. Algunas actividades correspondrán a escribir código y otras a responder preguntas.

**Importante.** Para facilitar su ejecución, cada sección puede ser ejecutada independientemente.

Se recomienda **fuertemente** revisar las secciones donde se entrega código porque algunas actividades de código pueden reutilizar el mismo código pero con cambios en algunas líneas.

## 1.0 Introduction

Generative Question Answering (GenQA) is a challenging task that NLP tries to solve. The aim is to provide solution to queries expressed in natural language automatically (Hovy, Gerber, Hermjakob, Junk, and Lin 2000). For instance, given the following context:

> Quito, oficialmente San Francisco de Quito, es la capital de la República del Ecuador, de la Provincia de Pichincha y la capital más antigua de Sudamérica. Es la ciudad más poblada del Ecuador,​ con 2 millones de habitantes en el área urbana, y aproximadamente 3 millones en todo el Área metropolitana.

We ask the question

> ¿Cuál es la población de Quito?

We expect the GenQA system generates something like this:

> 2 millones

Since 2017, transformer models have been shown to outperform existing approaches for this task. Currently, many pretrained transformer models exist, including GPT-2, XLNet, BART, T5, etc.

This tutorial shows how you can fine-tune BART/T5 for the task of GenQA and use it for inference. We will use the transformer library built by [Hugging Face](https://huggingface.co/), which is an extremely useful implementation of the transformer models in both TensorFlow and PyTorch. You can just use a fine-tuned model from their [model hub](https://huggingface.co/models).

This tutorial is for educational purposes with which we will learn to finetune a BART/T5 model and use it with your own data.

## Using T5 model for QA

<figure>
<center>
<img src='https://miro.medium.com/v2/resize:fit:1200/0*85tQJdCyxgiVSd4H.png' width="700" />
</center>
</figure>

*   Input is the $Question$ tokens and the $Paragraph$ tokens separated by space.
*   Encoder takes the input and Decoder generates an output text.
*   The probability of each word answer is computed at each decoding step.

## 2.0 Setup

First, we clone and install the Hugging Face transformer library from Github.

In [ ]:
!mkdir -p downloads \
&& cd downloads \
&& git clone --branch v4.52.4 --depth 1 https://github.com/huggingface/transformers.git \
&& cd transformers \
&& pip install -e .

In [ ]:
!cd downloads \
&& git clone https://github.com/vgaraujov/Seq2Seq-Spanish-PLMs.git
!pip install rouge_score

In [ ]:

!pip install evaluate==0.4.4
!pip install accelerate==1.8.0 -U
!pip install datasets==3.6.0 -U
# !pip install --upgrade torch torchvision torchaudio

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

## 3.0 Train Model

This is where we can train our own model.

### 3.1 Get Training and Evaluation Data

The Spanish Question Answering Corpus (SQAC) is an extractive QA dataset with no unanswerable questions. It is created from
texts extracted from the Spanish Wikipedia, encyclopedic articles, newswire articles from Wikinews, and the Spanish section of the AnCora corpus (Taulé, Martí, and Recasens, 2008), which is a mix from different newswire and literature sources.

Read more about this dataset here: https://arxiv.org/abs/2107.07253

Now get the Spanish SQAC. We use `datasets` library to load the data.



In [ ]:
from datasets import load_dataset

dataset = load_dataset("avacaondata/sqac_fixed")

In [ ]:
dataset

### 3.2 Dataset Exploration

Let's explore an example of the dataset. You need to change `id` variable if you want to change the example.

In [ ]:
id = 100

In [ ]:
dataset['validation'][id]['context']

In [ ]:
dataset['validation'][id]['question']

In [ ]:
dataset['validation'][id]['answers']

### 3.3 Run training (Optional)

We can now train the model with the training set. We will use BARTO or T5S, two renowned encoder-decoder architectures exclusively pre-trained on Spanish corpora. Read more details about these models here: https://arxiv.org/abs/2309.11259

**Notes about parameters:**

`per_gpu_train_batch_size` specifies the number of training examples per iteration per GPU.

`save_steps` specifies number of steps before it outputs a checkpoint file. I've increased it to save disk space.

`num_train_epochs` sets the number of epochs, two epochs are recommended. It's currently set to one for the purpose of time.

NOTE: it takes about 1 hour to train the model! If you don't want to wait this long, feel free to skip this step and use a pretrained model!

#### Fine-tune T5S

In [ ]:
!python downloads/Seq2Seq-Spanish-PLMs/scripts/generativeqa/run_generativeqa.py \
  --model_name_or_path vgaraujov/t5-base-spanish \
  --do_train \
  --do_eval \
  --dataset_name avacaondata/sqac_fixed \
  --context_column context \
  --question_column question \
  --answer_column answers \
  --output_dir /content/model_output \
  --max_source_length 480 \
  --max_target_length 32 \
  --per_device_train_batch_size 8 \
  --per_device_eval_batch_size 8 \
  --num_train_epochs 6 \
  --do_predict \
  --predict_with_generate \
  --save_strategy epoch \
  --predict_with_generate \
  --overwrite_output_dir

#### Fine-tune BARTO

In [ ]:
!python downloads/Seq2Seq-Spanish-PLMs/scripts/generativeqa/run_generativeqa.py \
  --model_name_or_path vgaraujov/bart-base-spanish \
  --do_train \
  --do_eval \
  --dataset_name avacaondata/sqac_fixed \
  --trust_remote_code \
  --context_column context \
  --question_column question \
  --answer_column answers \
  --output_dir /content/model_output \
  --max_source_length 480 \
  --max_target_length 32 \
  --per_device_train_batch_size 16 \
  --per_device_eval_batch_size 16 \
  --num_train_epochs 6 \
  --do_predict \
  --save_strategy epoch \
  --predict_with_generate \
  --overwrite_output_dir

## 4.0 Setup prediction code

Now we can use the Hugging Face library to make predictions using our model. Note that a lot of the code is pulled from `run_squad.py` in the Hugging Face repository, with all the training parts removed.


In [ ]:
# READER NOTE: If an error occurs, please restart sesion and try again.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import Dataset
from typing import List, Optional, Tuple

import numpy as np
import os

os.environ["WANDB_DISABLED"] = "true"

If you have trained your own mode, you need to change the flag `use_own_model` to `True`. However, in the case that you want to use a pre-trained model of the hub, you need to change the flag `use_own_model` to `False`, and define the model variable `model_name_or_path`.

In this tutorial, we will use an already fine-tuned model on SQAC.

In [ ]:
# READER NOTE: Set this flag to use own model, or use pretrained model in the Hugging Face repository
use_own_model = False
type_model = "t5"

if use_own_model:
  model_name_or_path = "/content/model_output"
else:
  if type_model == "t5":
    model_name_or_path = "mrm8488/spanish-t5-small-sqac-for-qa"
  elif type_model == "bart":
    model_name_or_path = "vgaraujov/bart-base-spanish-sqac"

model = AutoModelForSeq2SeqLM.from_pretrained(model_name_or_path)
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

This is a function borrowed from [`run_generativeqa.py`](https://github.com/vgaraujov/Seq2Seq-Spanish-PLMs/tree/main/scripts/generativeqa).  This modified code allows to run predictions we pass in directly as strings, rather .json format like the training/test set.

In [ ]:
def preprocess_squad_batch(
    examples,
    question_column: str,
    context_column: str,
    answer_column: str,
) -> Tuple[List[str], List[str]]:
    questions = examples[question_column]
    contexts = examples[context_column]
    answers = examples[answer_column]

    def generate_input(_question, _context):
        return " ".join(["question:", _question.lstrip(), "context:", _context.lstrip()])

    inputs = [generate_input(question, context) for question, context in zip(questions, contexts)]
    targets = [answer["text"][0] if len(answer["text"]) > 0 else "" for answer in answers]
    return inputs, targets

def preprocess_function(examples):
    inputs, targets = preprocess_squad_batch(examples, "question", "context", "answers")

    model_inputs = tokenizer(inputs, max_length=480, padding=False, truncation=True)
    # Tokenize targets with text_target=...
    labels = tokenizer(text_target=targets, max_length=32, padding=False, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

ignore_pad_token_for_loss = True
# Data collator
label_pad_token_id = -100 if ignore_pad_token_for_loss else tokenizer.pad_token_id
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=label_pad_token_id,
)

args = Seq2SeqTrainingArguments(
    output_dir="/tmp",
    # eval_strategy="steps",
    # eval_steps=100,
    # logging_strategy="steps",
    # logging_steps=100,
    # save_strategy="steps",
    # save_steps=200,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=True,
    metric_for_best_model="rouge1",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    # tokenizer=tokenizer,
    data_collator=data_collator,
)

def run_prediction(questions, context):
  data = []
  for question in questions:
    data.append({"question": question , "context": context, "answers": {'text': [''], 'answer_start': []}})

  predict_dataset = Dataset.from_list(data)
  predict_dataset = predict_dataset.map(
      preprocess_function,
      batched=True,
      remove_columns=predict_dataset.column_names,
      desc="Running tokenizer on prediction dataset",
  )

  predict_results = trainer.predict(predict_dataset, metric_key_prefix="predict")
  predictions = predict_results.predictions
  predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
  predictions = tokenizer.batch_decode(
      predictions, skip_special_tokens=True, clean_up_tokenization_spaces=True
  )
  predictions = [pred.strip() for pred in predictions]
  return predictions

## 5.0 Run predictions

Now for the fun part... testing out your model on different inputs. Pretty rudimentary example here. But the possibilities are endless with this function.

In [ ]:
context = "Bélgica, oficialmente Reino de Bélgica, es uno de los veintisiete Estados soberanos que forman la Unión Europea. Está situado en el noroeste europeo. El país cubre una superficie de 30 528 km²1​ y posee en 2023 una población de 11.754.004 Su capital y la conurbación más poblada es Bruselas, mientras que su ciudad (municipio) más poblada es Amberes."

questions = ["¿Cuál es la población de Bélgica?",
             "¿En qué parte de Europa esta ubicado?",
             "¿Cuál es la la ciudad más poblada?",
             "¿Cuál es la cápital de Alemania?"]

# Run method
predictions = run_prediction(questions, context)

# Print results
print("Results:")
for i, pred in enumerate(predictions):
  print(questions[i],pred)

## 6.0 Activity

Now is your turn. Use the code in Section 5.0 (previous section) to generate your own predictions. To do that, you must change the context variables and questions. (3 pts)


In [ ]:
# Your code here
# Aca debe estar el codigo de 5.0 con diferente context y question, ademas guardada la ejecucion mostrando la prediccion

---

Based on this tutorial and the class, set whether the following statements are `True` or `False`.


In [ ]:
#@title Experts annotated the Spanish SQuAD v2 dataset (1 pt)
answer = None #@param ["None","False", "True"] {type:"raw"}

In [ ]:
#@title BART and BERT use exactly the same underlying transformer architecture (1 pt)
answer = None #@param ["None","False", "True"] {type:"raw"}

In [ ]:
#@title Encoder-decoder models are pre-trained to reconstruct the input text (1 pt)
answer = None #@param ["None","False", "True"] {type:"raw"}